# ViRL · 생성/학습 분리 GRPO · Qwen3-VL-4B · A100

**기존 4B의 4회 생성 결과 → 파일럿 64문항 → 남은 문항 계속 학습 → 새 모델로 8회 생성 → 최대 3라운드**.
원본 Qwen3-VL-4B-Instruct에서 시작하며 teacher/SFT 가중치는 사용하지 않습니다.
`RUN_MODE='full'`이면 파일럿 관문 통과 후 자동으로 전체 학습까지 이어집니다. `pilot`이면 파일럿만 실행합니다.

### v2·v3 수정 (2026-11-20 검토 반영, 원본은 `ViRL_RoundGRPO_4B_A100_Train.ipynb`)
1. `LEGACY_PROVENANCE`를 제출 모델 `training_configuration.json`으로 지정(자동 탐색이 3개 폴더를 찾아 멈추던 문제).
2. 백분율 거부 검사 제거: "What percentage…" 정답 `80` ↔ 답 `80%` 같은 **맞는 답**을 오답 처리하고 있었음(기존 생성물에서 걸린 24개 전부). 정수 검사(`2016.0` vs `2015`)는 유지.
3. 잘린 응답만 오답인 그룹도 **유지**(잘린 응답은 계속 손실에서 제외 → 끝까지 맞힌 풀이만 강화). v1은 약 22%를 버렸음.
4. ViRL 검증을 문항당 **4샘플**로, 후보는 이전 최고보다 검증 샘플 정확도 **1%p 이상** 높을 때만 채택.
5. 2라운드부터 base가 못 푼 문항(주관식 0/4, 객관식 0~1/4)도 최대 600개 다시 생성.
6. Drive checkpoint 정리: 진행 중 checkpoint는 최근 2개만 남김(라운드 경계·후보 checkpoint는 유지).
7. 주관식 답 나열 방지: 최종 답에 `or`로 대안을 나열하면 모든 대안이 정답이어야 보상.
8. 출력 길이 감시: 모든 검증 결과에 출력 토큰 중앙값·평균을 기록하고, 후보 채택·파일럿 관문에서 base 대비 **잘림 비율 +3%p 초과** 또는 **출력 중앙값 30% 초과 감소**면 정확도와 상관없이 불합격(실행 전 고정).
9. `TRAIN_QUESTIONS` 1100 → 1200 (학습 가능 그룹 1,164개 전부 사용).
10. **(v3) 생성 속도:** v2 파일럿 실측 0.55~0.74샘플/초(16문항씩·동시 32요청) → 기존 4B 생성과 같게 256문항씩·vLLM 기본 동시 처리. 결과는 같고 속도만 약 3배. `RUN_TAG`는 `round_grpo_v3`.
11. **(v4) 라운드:** v3는 1라운드가 ViRL 검증 +1%p를 못 넘어 on-policy 라운드가 시작도 못 했음(base 897 vs 893/1,184). v4는 3라운드를 모두 진행(직전 라운드 adapter로 새로 생성)하고, 직전 라운드가 출력 길이 기준에 걸릴 때만 멈춤. 최종 선택은 마지막에 ViRL 검증 + 여유 기준으로.
12. **(v4) 학습률 5e-6 → 2e-5:** v3는 KL 약 0.002로 거의 움직이지 않았음. 보상은 검증을 마쳤으므로 상향.
13. **(v4) 잘린 응답도 오답으로 학습:** v3는 잘린 응답을 손실에서 빼서 루프 억제 신호가 없었고 잘림이 늘었음(79 → 96샘플). 단, 학습 생성 한도(6,144)가 평가(16,384)보다 짧아 긴 정상 풀이도 벌받을 수 있음 → 출력 길이 기준으로 감시.
`RUN_TAG`는 `round_grpo_v4`.

### 반영한 수정
- frozen old policy 확률비, clipping, 원본 base 기준 KL, 고정 길이 정규화 DR-GRPO 손실.
- 잘린 응답은 보상 0을 기록하되 **학습 손실에서는 제외**. 그룹 보상이 모두 같으면 학습하지 않습니다.
- 실제 backward/gradient·확률 정렬 검사, KL/clip/정책 변화량 중단 기준, optimizer 재개.
- 고정 ViRL 296 검증으로만 후보 선택. 같은 GPU에서 원본 base도 새로 검증합니다.
- 첫 라운드 **학습용 풀이 생성은 생략**합니다. 모델 검증을 위한 생성 비용은 별도입니다.

### 기존 기록의 중요한 한계
첨부한 생성 코드는 `text / output_tokens / finish_reason / seed`만 저장했습니다.
원래 토큰 ID·sampler log-probability는 없으므로 첫 라운드는 **검사된 재토큰화 + 원본 모델 raw 확률 재계산** 방식입니다.
텍스트 복원·길이·EOS 검사를 통과하지 못한 **문항 그룹 전체**는 제외합니다. 이것이 원래 토큰열의 완전 복원을 증명하지는 않습니다.
새 라운드부터는 원본 토큰 ID, prompt token IDs, vLLM V1 **raw** 확률을 저장합니다.
손실은 raw-policy clipped surrogate이며, 새 라운드는 vLLM/HF의 raw 확률 차이를 제한적으로 보정합니다.
평가와 같은 temperature/top-k/top-p/presence 설정의 모든 분포 변화를 정확히 보정한 unbiased on-policy 추정량이라고 주장하지 않습니다.

### 실행 환경과 검증 범위
Colab A100 **80GB 권장**. 학습 이미지 해상도는 원래 생성 조건을 유지하므로 40GB에서 임의 축소하지 않습니다.
40GB 학습은 사전 검사에서 메모리 부족으로 멈출 수 있습니다. 최종 MMMU 평가는 별도 **40GB 평가 노트북**을 사용하세요.
제작 환경에서는 CPU 로직·문법·기존 MMMU 900개 재채점을 검증했습니다. A100 학습 실행과 성능 향상은 아직 검증하지 않았습니다.
학습이 실행되지 않는 원인은 로그에 남기며, 실패 후 다음 단계를 자동 실행하지 않습니다.


## 1. Drive 연결과 실행 설정
기존 파일을 자동 탐색합니다. 다른 위치에 있거나 여러 파일이 다르면 아래 3개 경로만 지정하세요.
- 생성 파일: `virl_generations_PRIVATE.jsonl` 또는 `mixed_generations_PRIVATE.jsonl` (**정답 선별 파일/teacher 파일 아님**)
- 기존 누수 검사 통과 pool ZIP
- 원래 자기증류 실행 폴더의 `training_configuration.json` (생성 설정 확인용)

원본 파일을 덮어쓰지 않습니다. 설정을 변경한 실험은 새로운 `RUN_TAG`를 사용하세요.
같은 실험을 재개할 때는 동일한 설정과 RUN_TAG를 유지합니다.


In [1]:
import sys, json, os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

RUN_TAG = 'round_grpo_v4'   # v4: 3라운드 on-policy·lr 2e-5·잘린 응답 학습 (2026-11-20). v1~v3 폴더와 섞이지 않음
RUN_MODE = 'pilot'      # 'pilot': 파일럿만 / 'full': 파일럿 통과 후 전체 + 후속 라운드
WORK = Path('/content/virl_round_grpo')
ROOT = Path('/content/drive/MyDrive/VIRL39K_RFT_4B')
RUN_DIR = ROOT / RUN_TAG
WORK.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 자동 탐색이 실패할 때만 실제 Drive 경로를 입력합니다.
LEGACY_GENERATIONS = None
POOL_ZIP = None
# v2: 자동 탐색은 제출·lr1e-4·짧은 풀이 3개 폴더를 찾아 멈추므로 제출 모델 설정을 직접 지정
LEGACY_PROVENANCE = '/content/drive/MyDrive/VIRL39K_RFT_4B/virl39k_rft4b_bf16_lora_4adapters/training_configuration.json'

TRAIN_QUESTIONS = 1200     # 적격 문항이 적으면 실제 개수를 사용하며 보고서에 기록
PILOT_QUESTIONS = 64
MAX_ROUNDS = 3   # v4: 3라운드 모두 진행(직전 라운드 adapter로 새로 생성)
LEARNING_RATE = 2e-5   # v4: v3(5e-6)은 KL 약 0.002로 거의 안 움직임. 보상 검증 후 상향
assert RUN_MODE in ('pilot', 'full')
assert 1 <= MAX_ROUNDS <= 3
print('저장 위치:', RUN_DIR)


Mounted at /content/drive
저장 위치: /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4


## 실행 코드 펼치기
필요한 작업 코드·원본 채점기·평가 코드를 노트북 안에 넣었습니다. 별도 `.py` 업로드는 필요 없습니다.
이 셀은 코드를 풀어 놓기만 하며 학습이나 생성은 시작하지 않습니다.


In [2]:
import base64, hashlib, io, zipfile
BUNDLE_B64 = '...'
BUNDLE_SHA256 = '6caeb94cae4fa7d58b370fe678fc159b0f6462bd5acc80a6ab33e2ea5e36fda9'
payload = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(payload).hexdigest() == BUNDLE_SHA256
BUNDLE_DIR = WORK / ('code_' + BUNDLE_SHA256[:12])
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        dest = (BUNDLE_DIR / name).resolve()
        assert dest.is_relative_to(BUNDLE_DIR.resolve())
        data = z.read(name)
        if dest.exists():
            assert dest.read_bytes() == data, f'코드가 바뀌었습니다: {dest}'
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
from common import atomic_json, read_json, immutable_json
from orchestrate import run
print('코드:', BUNDLE_DIR)


코드: /content/virl_round_grpo/code_6caeb94cae4f


## 별도 실행 환경 설치
생성은 **vLLM 0.11.0**, 학습은 **Transformers 4.57.1 + PEFT 0.17.1 + PyTorch 2.8.0**입니다.
두 Python 3.12 환경과 각 작업 프로세스를 분리합니다. 한 GPU에 두 모델을 동시에 올리지 않습니다.
최초 설치와 모델 다운로드에는 시간이 듭니다. Colab 기본 Python 패키지를 교체하지 않습니다.


In [3]:
run(sys.executable, 'setup_env.py', ['--work', WORK], WORK / 'installation.log')
TRAIN_PY = str(WORK / 'train_env/bin/python')
GEN_PY = str(WORK / 'gen_env/bin/python')
run(TRAIN_PY, 'test_suite.py', [], WORK / 'cpu_tests.log')
print('설치 및 CPU/Torch 손실 검사 통과')


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: virl_round_grpo/train_env
Activate with: source virl_round_grpo/train_env/bin/activate
Using Python 3.12.3 environment at: virl_round_grpo/train_env
Resolved 71 packages in 360ms
Prepared 71 packages in 38.35s
Installed 71 packages in 222ms
 + accelerate==1.11.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + antlr4-python3-runtime==4.13.2
 + anyio==4.16.0
 + attrs==26.1.0
 + av==19.0.2
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + datasets==4.3.0
 + dill==0.4.0
 + filelock==4.2.12
 + frozenlist==1.8.0
 + fsspec==2025.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==0.36.0
 + idna==3.20
 + jinja2==3.1.6
 + latex2sympy2-extended==1.10.2
 + markupsafe==3.0.4
 + math-verify==0.8.0
 + mpmath==1.3.0
 + multidict==7.2.1
 + multiprocess==0.70.16
 + networkx==3.7
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu

## 2. 학습 설정 고정
한 update는 문항 4그룹을 사용하고, 각 completion은 microbatch 1로 처리합니다.
LoRA r=16 / alpha=32 / dropout=0, 언어 self-attention q/k/v/o만 학습합니다.
raw log-probability의 기준은 temperature 적용 전입니다. `old`는 해당 라운드의 생성 모델,
`reference`는 모든 라운드에서 **원본 base**로 고정합니다.

파일럿은 기술 검사 통과 + 고정 96문항에서 원본 대비 정확도 5%p 초과 하락/잘림 5%p 초과 증가가 없는지 확인합니다.
이 기준은 큰 이상을 찾기 위한 관문이며, 성능 향상의 통계적 증거가 아닙니다.
KL/clip 기준은 실험 시작값입니다. 기준에 걸리면 해당 update를 적용하지 않고 저장합니다.


In [4]:
CONFIG = {
    'run_dir': str(RUN_DIR), 'work_dir': str(WORK),
    'train_python': TRAIN_PY, 'gen_python': GEN_PY, 'seed': 3407,
    'input_roots': ['/content/drive/MyDrive/VIRL39K_RFT_4B', '/content/drive/MyDrive/MIXED_RFT_4B'],
    'legacy_generations': LEGACY_GENERATIONS, 'pool_zip': POOL_ZIP,
    'legacy_provenance': LEGACY_PROVENANCE,
    'expected_pool_rows': 5996, 'expected_val_rows': 296,
    'train_questions': TRAIN_QUESTIONS, 'min_train_questions': 128,
    'category_caps': {'(GradeSchool) Geometric': 350, '(GradeSchool) Non-Geo Math': 800},
    'pilot_questions': PILOT_QUESTIONS, 'pilot_val_questions': 96,
    'max_rounds': MAX_ROUNDS, 'fresh_samples': 8,
    'learning_rate': LEARNING_RATE, 'beta': 0.04, 'clip_epsilon': 0.2,
    'groups_per_update': 4, 'warmup_steps': 10,
    'loss_length_normalizer': 6144, 'max_grad_norm': 1.0,
    'max_reference_kl': 0.20, 'max_clip_fraction': 0.60, 'max_old_logp_drift': 0.20,
    'max_backend_logp_difference': 0.15,
    'checkpoint_every_steps': 10,
    'max_train_hours_per_invocation': 6.0, 'max_workflow_hours': 12.0,
    'max_pilot_accuracy_drop': 0.05, 'max_pilot_truncation_increase': 0.05,
    # v2 (2026-11-20 검토 반영)
    'val_samples': 4,                 # ViRL 검증 문항당 샘플 수(1샘플은 ±7문항 노이즈)
    'min_val_gain': 0.01,             # 후보 채택: 이전 최고보다 검증 샘플 정확도 1%p 이상
    'refresh_unsolved_max': 600,      # 2라운드부터 다시 생성할 base 미해결 문항 상한
    'keep_running_checkpoints': 2,    # Drive에 남길 진행 중 checkpoint 수
    'max_val_truncation_increase': 0.03,  # 후보·파일럿: base 대비 잘림 비율 +3%p 초과면 불합격
    'min_val_median_length_ratio': 0.70,  # 후보·파일럿: 출력 중앙값이 base의 70% 미만이면 불합격
    'gen_chunk_questions': 256,       # v3: vLLM 생성 묶음(문항 수). 기존 4B 생성과 같음
    'train_on_truncated': True,       # v4: 잘린 응답도 오답 쪽으로 학습(루프 억제). v3는 제외해 잘림이 늘었음
    'continue_rounds_without_val_gain': True,  # v4: 라운드 사이 검증 관문 없이 진행, 마지막에만 선택
}
assert 0 < PILOT_QUESTIONS <= TRAIN_QUESTIONS
assert PILOT_QUESTIONS % CONFIG['groups_per_update'] == 0
CONFIG_FILE = RUN_DIR / 'config.json'
immutable_json(CONFIG_FILE, CONFIG)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


{
  "run_dir": "/content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4",
  "work_dir": "/content/virl_round_grpo",
  "train_python": "/content/virl_round_grpo/train_env/bin/python",
  "gen_python": "/content/virl_round_grpo/gen_env/bin/python",
  "seed": 3407,
  "input_roots": [
    "/content/drive/MyDrive/VIRL39K_RFT_4B",
    "/content/drive/MyDrive/MIXED_RFT_4B"
  ],
  "legacy_generations": null,
  "pool_zip": null,
  "legacy_provenance": "/content/drive/MyDrive/VIRL39K_RFT_4B/virl39k_rft4b_bf16_lora_4adapters/training_configuration.json",
  "expected_pool_rows": 5996,
  "expected_val_rows": 296,
  "train_questions": 1200,
  "min_train_questions": 128,
  "category_caps": {
    "(GradeSchool) Geometric": 350,
    "(GradeSchool) Non-Geo Math": 800
  },
  "pilot_questions": 64,
  "pilot_val_questions": 96,
  "max_rounds": 3,
  "fresh_samples": 8,
  "learning_rate": 2e-05,
  "beta": 0.04,
  "clip_epsilon": 0.2,
  "groups_per_update": 4,
  "warmup_steps": 10,
  "loss_length_normalizer": 6144

## 3. 파일럿부터 실행
기존 split을 재구성하고 검증 문항과 이미지/질문이 정확히 겹치는 학습 문항을 제거합니다.
기존 pool의 MMMU/Pro 누수 검사를 전제로 하며, 여기서 의미적 유사 문항 검사까지 다시 하지는 않습니다.
객관식은 기존 v2, 주관식은 첨부 teacher 노트북의 v3를 내장했습니다.
v3의 `2016.0 vs 2015` 정수 오판을 막는 **거부 전용 검사**와 `or` 답 나열 방지를 추가하며(v2: 백분율 거부는 맞는 답을 깎아 제거),
원래 v3 점수와 실제 보상을 모두 기록합니다. 세부 내역은 `data/reward_audit_PRIVATE.jsonl`입니다.
최종 MMMU 채점에는 이 검사나 open-v3를 적용하지 않습니다.

`full`은 파일럿의 가중치·optimizer를 이어 받아 **이미 학습한 파일럿 문항 다음부터** 진행합니다.
후속 라운드는 이전 라운드가 ViRL 296에서 새 최고 정확도를 얻었을 때만 시작합니다.
동점은 이전 모델을 유지합니다. 새 최고가 없으면 원본 base가 선택될 수 있습니다.


In [5]:
run(sys.executable, 'orchestrate.py', ['--config', CONFIG_FILE, '--mode', RUN_MODE],
    RUN_DIR / ('notebook_' + RUN_MODE + '.log'))


Audit 100/5996 | eligible 23
Audit 600/5996 | eligible 147
Audit 700/5996 | eligible 166
Audit 2700/5996 | eligible 528
Audit 3100/5996 | eligible 600
Audit 3900/5996 | eligible 765
Audit 4000/5996 | eligible 782
Audit 4700/5996 | eligible 957
Audit 5600/5996 | eligible 1079
{
  "pool_rows": 5996,
  "train_groups": 1160,
  "val_rows": 296,
  "legacy_generations": "/content/drive/MyDrive/VIRL39K_RFT_4B/virl_generations_PRIVATE.jsonl",
  "legacy_provenance": "/content/drive/MyDrive/VIRL39K_RFT_4B/virl39k_rft4b_bf16_lora_4adapters/training_configuration.json",
  "rejected_groups": {
    "not_partial_success": 4368,
    "exact_val_overlap": 149,
    "Legacy text/token-count mismatch": 22,
    "Stopped completion has no recoverable EOS": 1
  },
  "categories": {
    "Tables/Diagrams/Charts": 331,
    "(GradeSchool) Non-Geo Math": 241,
    "(GradeSchool) Science": 183,
    "Spatial Reasoning": 149,
    "(GradeSchool) Geometric": 122,
    "Social Science": 86,
    "Broader STEM Topics": 48
  

2046.42345045

## 4. 결과 확인
`pilot`로 시작했다면 아래 셀에서 전체 학습을 이어갈 수 있습니다.
Colab 연결 종료 후에는 위 셀을 같은 설정으로 다시 실행하면 저장된 optimizer 경계에서 재개합니다.
마지막 저장 이후의 작업은 다시 수행될 수 있지만, 저장된 update가 중복 적용되지는 않습니다.
KL/clip 중단은 같은 오래된 풀이로 강행하지 않습니다. 후보 검증을 거쳐 후속 라운드에서 새 풀이를 생성합니다.
시간 제한은 안전한 경계에서 적용하는 소프트 제한이며 설치·검증·병합 시간은 별도입니다.


In [6]:
for rel in ['data/report.json', 'pilot_gate.json', 'selection_report.json']:
    p = RUN_DIR / rel
    if p.exists():
        print('\n' + rel)
        print(json.dumps(read_json(p), ensure_ascii=False, indent=2))
handoff = RUN_DIR / 'eval_handoff.json'
print('\n40GB 평가 노트북 HANDOFF_FILE:', str(handoff) if handoff.exists() else '전체 학습 후 생성됩니다.')



data/report.json
{
  "pool_rows": 5996,
  "train_groups": 1160,
  "val_rows": 296,
  "legacy_generations": "/content/drive/MyDrive/VIRL39K_RFT_4B/virl_generations_PRIVATE.jsonl",
  "legacy_provenance": "/content/drive/MyDrive/VIRL39K_RFT_4B/virl39k_rft4b_bf16_lora_4adapters/training_configuration.json",
  "rejected_groups": {
    "not_partial_success": 4368,
    "exact_val_overlap": 149,
    "Legacy text/token-count mismatch": 22,
    "Stopped completion has no recoverable EOS": 1
  },
  "categories": {
    "Tables/Diagrams/Charts": 331,
    "(GradeSchool) Non-Geo Math": 241,
    "(GradeSchool) Science": 183,
    "Spatial Reasoning": 149,
    "(GradeSchool) Geometric": 122,
    "Social Science": 86,
    "Broader STEM Topics": 48
  },
  "reward_version": "original MC-v2 / open-v3 + rejecting-only numeric guards",
  "legacy_limitation": "Original token IDs/sampler probabilities absent. Roundtrip/count checked; raw-policy likelihoods reconstructed, not exact behavior-policy replay.",
  "

## 5. 파일럿만 실행한 뒤 전체 학습으로 이어가기
기본 `full` 실행에서는 이미 처리되었으므로 건너뜁니다.


In [8]:
CONTINUE_AFTER_PILOT = True
if RUN_MODE == 'pilot' and CONTINUE_AFTER_PILOT:
    try:
        run(sys.executable, 'orchestrate.py', ['--config', CONFIG_FILE, '--mode', 'full'], RUN_DIR / 'notebook_full.log')
    finally:
        from google.colab import runtime
        print('끝(또는 오류). 60초 뒤 런타임을 종료합니다.')
        import time; time.sleep(60)
        runtime.unassign()
else:
    print('추가 실행 없음. full 모드가 시간 제한으로 멈췄다면 3번 셀을 다시 실행하세요.')


Reusing frozen train/validation data: 1160 296

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.33s/it]
step=17 groups=68/1160 loss=0.04899 KL=0.0005 clip=0.002
step=18 groups=72/1160 loss=0.01412 KL=0.0004 clip=0.002
step=19 groups=76/1160 loss=0.06393 KL=0.0005 clip=0.003
step=20 groups=80/1160 loss=0.00180 KL=0.0004 clip=0.004
step=21 groups=84/1160 loss=0.03634 KL=0.0006 clip=0.003
step=22 groups=88/1160 loss=0.02251 KL=0.0011 clip=0.003
step=23 groups=92/1160 loss=0.00579 KL=0.0006 clip=0.003
step=24 groups=96/1160 loss=0.02241 KL=0.0008 clip=0.004
step=25 groups=100/1160 loss=0.07563 KL=0.0007 clip=0.003
step=26 groups=104/1160 loss=0.06121 KL=0.0008 clip=0.005
step=27 groups=108/1160 loss=0.03210 KL=0.0006 clip=0.004
step=28 groups=112/1160 loss=0.00059 KL=0.0073 clip=0.128
step=29 groups=116/1160 loss=0.02844 KL=0.0011 clip=0.007
step=30 groups=120/1160 loss=0.00448 KL=0.0008 clip=0.006
step=31 groups=124/1160 loss=-0.00550 KL=0.0010 clip=0.009
step=32 groups=1

RuntimeError: orchestrate.py failed (1). Check /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4/notebook_full.log. No later stage was launched.

### 기록과 한계
- `data/report.json`: 실제 학습 문항 수, 토큰 복원 탈락, 채점 검사, 분야 분포.
- `round_*/train/metrics.jsonl`: loss/KL/clipping/gradient/시간. `latest.json`: 재개 위치.
- `validation/`: 같은 조건에서 새로 평가한 base·pilot·각 라운드 결과.
- `selection_report.json`, `eval_handoff.json`: ViRL로만 선택한 후보와 40GB 평가 연결 정보.
- `stage_timing.jsonl`: 단계별 실측 시간. Colab 유닛 비용과 성능 상승 확률은 임의로 추정하지 않습니다.
- 검증 점수가 오르더라도 MMMU로의 전이는 별도입니다. 실패한 실험도 같은 기준으로 기록하세요.

참고: [TRL GRPO](https://huggingface.co/docs/trl/v0.24.0/grpo_trainer),
[DAPO의 잘림 처리](https://arxiv.org/html/2503.14476v1#S3.SS4),
[vLLM 0.11 V1 raw log-probability](https://github.com/vllm-project/vllm/blob/v0.11.0/vllm/v1/sample/sampler.py).
